# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
# Install the datasets library if needed
!pip install -q huggingface_hub

from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF-TOKEN')
login(token=hf_token)

In [22]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
fs.ls("datasets/FlyRank/internship-warehouse", detail=False)

['datasets/FlyRank/internship-warehouse/fact_content_daily_performance',
 'datasets/FlyRank/internship-warehouse/.gitattributes',
 'datasets/FlyRank/internship-warehouse/README.md',
 'datasets/FlyRank/internship-warehouse/dim_clients.parquet',
 'datasets/FlyRank/internship-warehouse/dim_content.parquet',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet',
 'datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet']

In [23]:
fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance", detail=False)

['datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-09',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-10',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-11',
 'datasets/FlyRank/internship-warehouse/fac

In [24]:
fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03", detail=False)

['datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet']

In [25]:
import pandas as pd

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(df.shape)
df.head()

(9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [26]:
#One row = one (report_date, client, content_item) combination — a daily performance record for a single piece of content, for a single client


## 2. Fields: feature / label / context / excluded

**Feature** (safe to use — knowable before the decision moment):
- `gsc_impressions` — how many times the page appeared in search results
- `gsc_clicks` — how many times the page was clicked
- `gsc_sum_position` — used to compute average search ranking position
- `client_has_gsc` — whether this client has Search Console connected
- `client_has_ga4` — whether this client has Analytics connected

**Label / proxy:**
None — clustering (Lane 3) is unsupervised. There is no target being predicted, only groupings
discovered from the features above.

**Context** (grouping/joining/filtering only — never fed to the model):
- `client_hash_id` — pseudonymized client ID, used only for grouping
- `content_hash_id` — pseudonymized content ID, used only for grouping
- `report_date` — defines the time window/slice, not a learned signal
- `month` — partition marker, not a learned signal

**Excluded** (with reason):
- `gsc_data_available`, `ga4_data_available` — these are filtering flags used to check data
  quality, not performance signals about the content itself
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`,
  `sessions_ai`, `scroll_events` — mostly NaN in this slice (many clients have
  `client_has_ga4 = False`); including them risks the model learning a "missingness pattern"
  instead of a real signal

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
#Grain
dupes = df.groupby(["report_date", "client_hash_id", "content_hash_id"]).size()
print("Rows with duplicates:", (dupes > 1).sum())





Rows with duplicates: 0


In [28]:
#Counts + date span
print("Total rows:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())
print("Unique clients:", df["client_hash_id"].nunique())
print("Unique content items:", df["content_hash_id"].nunique())

Total rows: 9841378
Date range: 2026-03-01 to 2026-03-31
Unique clients: 55
Unique content items: 331437


In [29]:
#Availability check with IS TRUE
before = len(df)
available = df[df["gsc_data_available"] == True]
after = len(available)
print(f"Rows before filter: {before}")
print(f"Rows after gsc_data_available IS TRUE: {after}")
print(f"Dropped: {before - after} ({(before-after)/before:.1%})")

Rows before filter: 9841378
Rows after gsc_data_available IS TRUE: 3611061
Dropped: 6230317 (63.3%)


In [30]:
# Start from the FILTERED dataframe (gsc_data_available == True)
available = df[df["gsc_data_available"] == True]

# Aggregate daily rows up to one row per content item for the month
features = available.groupby("content_hash_id").agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_sum_position", lambda x: x.sum() / available.loc[x.index, "gsc_impressions"].sum()),
    days_with_data=("report_date", "nunique"),
    client_has_ga4=("client_has_ga4", "first")
).reset_index()

print(features.shape)
features.head()

(176738, 6)


,content_hash_id,total_impressions,total_clicks,avg_position,days_with_data,client_has_ga4
0,content_000005d4ced12088,86,0,72.081395,24,True
1,content_00007bd2985b77c3,47,0,5.297872,23,False
2,content_0000cd28fbda69f3,29,0,3.827586,13,False
3,content_0000d495bfbfb4a8,15,0,2.133333,4,True
4,content_00014efc121d911d,116,1,5.793103,30,False


In [31]:
# Step 1: Create a proxy label — is this a high-performing page? (top 25% by clicks)
features["is_top_performer"] = (features["total_clicks"] >= features["total_clicks"].quantile(0.75)).astype(int)

# Step 2: THE TRAP — add a leaky feature derived directly from the label
# click_rank_leaky is computed FROM total_clicks, the same thing the label is based on
features["click_rank_leaky"] = features["total_clicks"].rank(pct=True)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# WITH the leak
leaky_feats = ["total_impressions", "avg_position", "days_with_data", "click_rank_leaky"]
X_leaky = StandardScaler().fit_transform(features[leaky_feats])
km_leaky = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_leaky)

# Check how well the "leaky" clustering lines up with the label (too well = leakage)
from sklearn.metrics import adjusted_rand_score
leaky_score = adjusted_rand_score(features["is_top_performer"], km_leaky.labels_)
print("WITH leak (click_rank_leaky included) — alignment with label:", round(leaky_score, 3))

WITH leak (click_rank_leaky included) — alignment with label: 0.639


In [32]:
# Step 3: Remove the leaky feature, keep only honest ones
honest_feats = ["total_impressions", "avg_position", "days_with_data"]
X_honest = StandardScaler().fit_transform(features[honest_feats])
km_honest = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_honest)

honest_score = adjusted_rand_score(features["is_top_performer"], km_honest.labels_)
print("WITHOUT leak (honest features only) — alignment with label:", round(honest_score, 3))

WITHOUT leak (honest features only) — alignment with label: 0.055


**Five features for Lane 3 (clustering), aggregated to one row per content item:**

1. **total_impressions** — sum of search impressions over the month.
   Knowable at the decision moment because it's a fully observed, historical count —
   no future information is needed to compute it.

2. **total_clicks** — sum of clicks over the month.
   Knowable at the decision moment because, like impressions, it's a completed historical
   record for the window being analyzed.

3. **avg_position** — impression-weighted average search ranking position.
   Knowable at the decision moment because it's derived purely from already-observed
   gsc_sum_position and gsc_impressions values, no future data involved.

4. **days_with_data** — number of distinct days in the month this content had available data.
   Knowable at the decision moment because it just counts observed reporting days,
   a fact about data coverage, not a future outcome.

5. **client_has_ga4** — whether the client has Analytics connected.
   Knowable at the decision moment because this is a static account-level property,
   known well before this or any future reporting period.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



A major limitation of this slice: filtering to `gsc_data_available IS TRUE` drops 63.3% of
rows (6,230,317 of 9,841,378 daily rows for March 2026), leaving only 3,611,061 usable rows
before aggregation. This means any features or clusters I build are only representative of
the roughly 37% of content-days with confirmed Search Console data — not the full content
inventory tracked in the warehouse.

I don't know from this slice alone whether missing availability is random (e.g. a temporary
sync gap) or systematic (e.g. certain clients or content types consistently lack GSC data).
If it's systematic, my clusters could be biased toward whichever clients/content happen to
have reliable tracking, and could misrepresent or completely miss patterns among the excluded
majority. This is a real gap in this contract, not just a technical footnote — future work
should check whether availability correlates with client, content type, or content age before
treating cluster results as representative of the whole inventory.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.